In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from cavity_correction import correct_cavity
from prefilter_correction import correct_prefilter
from ghost_correction import correct_ghost
from fringe_correction import correct_fringes
from crosstalk_correction import correct_crosstalk
from processing import *
from classical_estimates import classical_estimates

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

['/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240330T050009_V202608262158C_0463300100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240926T114503_V202608262134C_0469260100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241016T113003_V202608262111C_0470160100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241027T233003_V202608262048C_0470270100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241202T123003_V202608262027C_0472020100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250119T210009_V202608262003C_0561190100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250310T080009_V202608261939C_0563100100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250923T000503_V202608261853C_0569230100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20260310T040003_V202608261828C_0663100

In [4]:
i = -2
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [5]:
folder = '/home/ulyanov/data/solo/phi/202607/data/'
folder_out = '/home/ulyanov/data/solo/phi/202607/blos_/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260701T030009_V202607061032C_0647010501.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260701T070009_V202607061032C_0647010502.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260701T110009_V202607061133C_0647010503.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260701T150009_V202607061133C_0647010504.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260701T190009_V202607061133C_0647010505.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260701T230009_V202607061133C_0647010506.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260702T030009_V202607061133C_0647020501.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260702T070009_V202607061133C_0647020502.fits.gz',
 '/home/ulyanov/data/solo/phi/202607/data/solo_L1_phi-fdt-alam_20260702T110009_V

In [68]:
for file in files[:1]:

    data, header = process(file,
                           dark_file=dark_file,
                           prefilter_file=prefilter_file,
                           #cavity_file=cavity_file,
                           flatfield_file=flat_file,
                           ghost_file=ghost_file,
                           distortion_file=distortion_file,
                           _realign=True,
                           _find_center=True,
                           _demodulate=True,
                           _correct_fringes=True,
                           _correct_crosstalk=True,
                           )

    Blos, Vlos = classical_estimates(data, header)

    #file_out = generate_filename(file, prefix='blos')
    #hdul = fits.HDUList([fits.PrimaryHDU(data=Blos.clip(-1e4,1e4).astype(np.float32), header=header)])
    #hdul.writeto(folder_out + '/' + file_out, overwrite=True)

In [65]:
plt.figure(figsize=(10,10))
plt.imshow(data[2,3], 'gray', vmin=-30, vmax=30)
plt.tight_layout()

In [48]:
plt.figure(figsize=(10,10))
plt.imshow(data[2,0], 'gray')
plt.tight_layout()

In [49]:
plt.figure(figsize=(10,10))
plt.imshow(Blos, 'seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [50]:
plt.figure(figsize=(10,10))
plt.imshow(Vlos, 'seismic', vmin=-3000, vmax=3000)
plt.tight_layout()

In [69]:
xc, yc = 600,300
h = 100

temp = data[:,3,xc-h:xc+h,yc-h:yc+h]
temp = np.median(temp, axis=(-2,-1))

plt.figure(figsize=(10,8))
plt.plot(temp)
plt.tight_layout()